In [1]:
import pandas as pd
import numpy as np
import time
import chess
import chess.engine
import asyncio
from pathfinding.core.diagonal_movement import DiagonalMovement
from pathfinding.core.grid import Grid
from pathfinding.finder.a_star import AStarFinder
from IPython.display import clear_output, display

In [2]:
def tabuleiro_grade37x37():
    # Gerar Nomes das Colunas (Margens + Peças + Gaps = 37 colunas)
    # 3 no início
    colunas = ['E0', 'E1', 'E2'] 
    letras = ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']
    p_count = 1
    
    for i in range(8):
        # Insere o bloco da peça (ex: a0, a, a1)
        colunas.extend([f"{letras[i]}0", letras[i], f"{letras[i]}1"])
        
        # Insere o gap de 1 coluna (p1, p2, p3...), exceto após a coluna 'h'
        if i < 7:
            colunas.extend([f"p{p_count}"])
            p_count += 1
            
    # no final
    colunas.extend(['D0', 'D1', 'D2'])
    
    # Gera Nomes das Linhas (Idêntico às colunas, usando L1, L2...)
    linhas = ['Top0', 'Top1', 'Top2']
    numeros = ['8', '7', '6', '5', '4', '3', '2', '1']
    l_count = 1
    
    for i in range(8):
        # Bloco da peça (ex: 8_0, 8, 8_1)
        linhas.extend([f"{numeros[i]}_0", numeros[i], f"{numeros[i]}_1"])
        
        # Gap de 1 linhas (L1, L2, L3...)
        if i < 7:
            linhas.extend([f"L{l_count}"])
            l_count += 1
            
    linhas.extend(['Bot0', 'Bot1', 'Bot2'])
    
    # Criar o DataFrame 37x37 preenchido com ""
    matriz_base = np.full((37, 37), '', dtype='<U4')
    tabuleiro = pd.DataFrame(matriz_base, index=linhas, columns=colunas)

    return tabuleiro

def preencher_tabuleiro(grid37x37,fen):
    tabuleiro = grid37x37
    tabuleiro[:] = "."
    linhas = tabuleiro.index.tolist()
    colunas = tabuleiro.columns.tolist()

    # le as posições das pecas no tabuleiro 8x8 (FEN)
    posicao = fen.split(' ')[0]
    linhas_fen = posicao.split('/')
    
    # Distribui as peças com o novo espaçamento
    nu_linha = 0
    for l in linhas_fen:
        nu_colum = 0

        # 4 * posição 
        novo_l = (4 * nu_linha) + 4
        for c in l:
            
            if c.isdigit():
                nu_colum += int(c)

            else:
                novo_c = (4 * nu_colum) + 4
                # Preenche ao redor com '#' e o centro com a peca
                for i in range(-1,2,1):
                    for j in range(-1,2,1):
                        # hoje aprendi que i==0 & j==0 é diferente de (i==0) & (j==0)
                        if i == 0 and j == 0:
                            tabuleiro.at[linhas[novo_l+i], colunas[novo_c +j]] = c
                        else:
                            tabuleiro.at[linhas[novo_l+i], colunas[novo_c +j]] = '#'

                        
                nu_colum += 1              
        nu_linha += 1
    return tabuleiro


In [3]:
#facil demais usando a biblioteca pathfinding. Talvez eu tente implementar o codigo eu mesmo
# para ter algum desafio.
def calcular_rota_a_star(tabuleiro_df, coordenada_inicio, coordenada_fim):

    matriz_str = tabuleiro_df.to_numpy()
    
    #  Onde for '.' vira 1 (Livre), o resto vira 0
    matriz_binaria = np.where(matriz_str == '.', 1, 0).tolist()
    
    # Inicio e fim precisa estar livre
    y_ini, x_ini = coordenada_inicio
    y_fim, x_fim = coordenada_fim
    
    matriz_binaria[y_ini][x_ini] = 1
    matriz_binaria[y_fim][x_fim] = 1 
    
    # Grid da biblioteca
    grid = Grid(matrix=matriz_binaria)
    
    # (coluna, linha) e nao (linha, coluna) como np.array
    start = grid.node(x_ini, y_ini)
    end = grid.node(x_fim, y_fim)
    # tem 4 opçoes para configurar a diagonal
    #   always = 1
    #    never = 2
    #    if_at_most_one_obstacle = 3
    #    only_when_no_obstacle = 4
    finder = AStarFinder(diagonal_movement=DiagonalMovement.if_at_most_one_obstacle)
    
    caminho, execucoes = finder.find_path(start, end, grid)
    
    # Retorna a lista de coordenadas (X, Y) do caminho
    return caminho
grade = tabuleiro_grade37x37()

fen = 'rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq'
tabuleiro = preencher_tabuleiro(grade,fen)
# (linha, coluna)
inicio = (6, 4)
destino = (34, 20)

rota = calcular_rota_a_star(tabuleiro, inicio, destino)

print(f"Caminho encontrado ({len(rota)} passos):")
print(rota)

Caminho encontrado (31 passos):
[<GridNode(4:6 0x7fcf836f1dc0)>, <GridNode(5:6 0x7fcf836f1df0)>, <GridNode(6:7 0x7fcf836f2510)>, <GridNode(6:8 0x7fcf836f2c00)>, <GridNode(6:9 0x7fcf836f32f0)>, <GridNode(7:10 0x7fcf836f3a10)>, <GridNode(8:11 0x7fcf83704170)>, <GridNode(8:12 0x7fcf83704860)>, <GridNode(8:13 0x7fcf83704f50)>, <GridNode(8:14 0x7fcf83705640)>, <GridNode(8:15 0x7fcf83705d30)>, <GridNode(8:16 0x7fcf83706420)>, <GridNode(8:17 0x7fcf83706b10)>, <GridNode(9:18 0x7fcf83707230)>, <GridNode(10:19 0x7fcf83707950)>, <GridNode(11:20 0x7fcf835180b0)>, <GridNode(12:21 0x7fcf835187d0)>, <GridNode(13:22 0x7fcf83518ef0)>, <GridNode(14:23 0x7fcf83519610)>, <GridNode(15:24 0x7fcf83519d30)>, <GridNode(16:25 0x7fcf8351a450)>, <GridNode(17:26 0x7fcf8351ab70)>, <GridNode(18:27 0x7fcf8351b290)>, <GridNode(18:28 0x7fcf8351b980)>, <GridNode(18:29 0x7fcf8352c0b0)>, <GridNode(18:30 0x7fcf8352c7a0)>, <GridNode(18:31 0x7fcf8352ce90)>, <GridNode(18:32 0x7fcf8352d580)>, <GridNode(18:33 0x7fcf8352dc70)>, 

In [4]:
caminho_limpo = [(node.x, node.y) for node in rota]

print("Coordenadas limpas (X, Y):")
print(caminho_limpo)

Coordenadas limpas (X, Y):
[(4, 6), (5, 6), (6, 7), (6, 8), (6, 9), (7, 10), (8, 11), (8, 12), (8, 13), (8, 14), (8, 15), (8, 16), (8, 17), (9, 18), (10, 19), (11, 20), (12, 21), (13, 22), (14, 23), (15, 24), (16, 25), (17, 26), (18, 27), (18, 28), (18, 29), (18, 30), (18, 31), (18, 32), (18, 33), (19, 34), (20, 34)]


In [5]:
def visualizar_rota(tabuleiro_df, caminho_nodes):
    # Cria uma cópia para não estragar o tabuleiro original
    tab_visual = tabuleiro_df.copy()
    
    linhas_reais = tab_visual.index.tolist()
    colunas_reais = tab_visual.columns.tolist()
    
    # Substitui cada passo da rota por um '*'
    for node in caminho_nodes:
        # Extrai X (coluna) e Y (linha)
        x_col = node.x
        y_lin = node.y
        
        # mapeia de volta para os nomes originais (ex: 'a', 'p1', '4_0')
        nome_linha = linhas_reais[y_lin]
        nome_coluna = colunas_reais[x_col]
        
        # Só marca com '*' se não for a própria peça (para não apagar a letra da peça)
        if tab_visual.at[nome_linha, nome_coluna] == '.':
            tab_visual.at[nome_linha, nome_coluna] = '*'
            
    return tab_visual

# teste
tabuleiro_com_rota = visualizar_rota(tabuleiro, rota)
print(tabuleiro_com_rota.to_string())

     E0 E1 E2 a0  a a1 p1 b0  b b1 p2 c0  c c1 p3 d0  d d1 p4 e0  e e1 p5 f0  f f1 p6 g0  g g1 p7 h0  h h1 D0 D1 D2
Top0  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
Top1  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
Top2  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
8_0   .  .  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  .  .
8     .  .  .  #  r  #  .  #  n  #  .  #  b  #  .  #  q  #  .  #  k  #  .  #  b  #  .  #  n  #  .  #  r  #  .  .  .
8_1   .  .  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  .  .  .
L1    .  .  .  .  *  *  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .  .
7_0   .  .  .  #  #  #  *  #  #  #  .  #  #  #  .  #  #  #  .  #  #  #  

In [6]:
grade = tabuleiro_grade37x37()
tabuleiro = preencher tabuleiro

SyntaxError: invalid syntax (3588339482.py, line 2)

In [ ]:
engine = chess.engine.SimpleEngine.popen_uci(r"/home/eros/virtualenvs/Chess/Stockfish/src/stockfish")
board = chess.Board()


In [ ]:
 async def main() -> None:
     transport, engine = await chess.engine.popen_uci(r"/home/eros/virtualenvs/Chess/Stockfish/src/stockfish")

     board = chess.Board()
     while not board.is_game_over():
         display(board)
         jogador = input()
         if not board.is_legal(chess.Move.from_uci(jogador)):
             clear_output(wait=True)
             continue
         # Jogada da pessoa 
         board.push_san(jogador)
         
         clear_output(wait=True)
         display(board)
         bot = await engine.play(board, chess.engine.Limit(depth = 10, time=10))
         # Jogada do bot
         board.push(bot.move)
         clear_output(wait=True)
         display(board)
         await asyncio.sleep(1)
         clear_output(wait=True)

     await engine.quit()

 await main()